# TTS Corpus Ingest

Reads a real speech corpus and writes a manifest of `(text, audio)` rows in the format that
[docs/manifest.md](../docs/manifest.md) defines. This is step 2b of the TTS dataset pipeline
described in [docs/pipeline.md](../docs/pipeline.md): an alternative to donor TTS synthesis
([`tts_dataset_gen.ipynb`](tts_dataset_gen.ipynb)) that takes its audio from an existing
recording instead of synthesising it.

### Pipeline stages

| # | Stage | Description |
|---|---|---|
| 0 | Configuration | Set the corpus kind, its inputs, and the speaker and hours rules |
| 1 | Load corpus rows | Read `(id, text, speaker, audio)` from a Hugging Face dataset, an LJSpeech directory, or a CSV |
| 2 | Speaker accounting | A first pass sums audio duration and row count per speaker |
| 3 | Speaker selection | Keep an explicit speaker list, or the top N by hours |
| 4 | Text validation | Refuse rows with empty text, a bracketed annotation, or no letter |
| 5 | Write the manifest | A second pass decodes audio, asserts mono, applies the hours cap, and writes WAV files |
| 6 | Manifest, sidecar and summary | Write `manifest.jsonl`, its `.meta.json`, and print the counts |

| Field | Value |
|---|---|
| Inputs | `CORPUS_KIND=hf` (a Hugging Face dataset, streamed), `CORPUS_KIND=ljspeech` (a local directory with `metadata.csv` and a `wavs/` folder), or `CORPUS_KIND=csv` (a local CSV with configurable text and audio-path columns); optional speaker selection (`SPEAKERS` or `TOP_SPEAKERS`) and an hours cap (`MAX_HOURS`). |
| Outputs | `OUTPUT_DIR/manifest.jsonl` (one row per kept clip, `source.kind` set to `corpus`, `speaker` set where the corpus names one), `OUTPUT_DIR/manifest.jsonl.meta.json` (the corpus id, its config and splits, the Hub revision read, the speaker rule, and the row counts), and `OUTPUT_DIR/wavs/` (mono PCM WAV at each clip's own native sample rate). |
| Runtime | SMOKE finishes in seconds: the built-in CSV fixture never reaches the network. A full `CORPUS_KIND=hf` run's time follows the dataset's size — it streams the corpus twice (an accounting pass, then the write pass), and each pass can pull up to a few hundred megabytes. |
| Hardware | CPU only. There is no GPU step and no ONNX model — this notebook reads a corpus, refuses bad text, and writes the manifest and audio; scoring is the measure notebook's job. |
| Cost | None: standard library plus `datasets`/`huggingface_hub`, no paid compute or API calls. `CORPUS_KIND=hf` reads a public Hugging Face dataset over outbound network access, not a paid service. |

CI: linted, compiled under Python 3.11 and 3.12, and executed on the smoke path.

Change the settings in the Settings cell, and then run all cells. To change a setting without a change to the notebook, set an environment variable with the same name. You can also write the line `NAME=value` in a file called `.env`, in the folder where the notebook runs.


In [ ]:
#@title Code that reads the settings
# A setting comes from an environment variable of the same name, else from a line NAME=value in the file .env,
# else from the value in the Settings cell. A value from outside is turned into the type of that value.
import json, os

_DOTENV = {}
if os.path.isfile(".env"):
    with open(".env", encoding="utf-8") as _f:
        for _line in _f:
            _name, _sep, _value = _line.strip().partition("=")
            if _sep and not _name.startswith("#"):
                _DOTENV[_name.strip()] = _value.strip().strip("\"'")


def setting(name, default):
    raw = os.environ.get(name, _DOTENV.get(name))
    if raw is None:
        return default
    if isinstance(default, bool):
        if raw.strip().lower() in ("1", "true", "yes", "on"):
            return True
        if raw.strip().lower() in ("0", "false", "no", "off", ""):
            return False
        raise ValueError(f"{name} must be True or False, not {raw!r}")
    if isinstance(default, (int, float)):
        try:
            return type(default)(raw)
        except ValueError:
            raise ValueError(f"{name} must be a number, not {raw!r}") from None
    if isinstance(default, (list, dict)):
        return json.loads(raw)
    return raw


In [ ]:
# Settings: change the values below.
CORPUS_KIND = setting("CORPUS_KIND", "hf")             # where the recordings are: hf (Hugging Face), ljspeech or csv (a folder)
CORPUS_LANG = setting("CORPUS_LANG", "en")             # the language and country of the recordings, for example pt-PT
OUTPUT_DIR = setting("OUTPUT_DIR", "")                 # the folder for the result; empty: corpus_ingest_out
CORPUS_NAME = setting("CORPUS_NAME", "")               # a name for the recordings; empty: the name of the dataset or file
CORPUS_URL = setting("CORPUS_URL", "")                 # a web page of the recordings; empty: the Hugging Face page, for hf
MANIFEST_NAME = setting("MANIFEST_NAME", "manifest.jsonl")  # the name of the list of clips that the notebook writes

# ── CORPUS_KIND = "hf": a dataset on Hugging Face ──
HF_DATASET = setting("HF_DATASET", "")                 # its name, for example "org/dataset-name"
HF_CONFIG = setting("HF_CONFIG", "")                   # its configuration; empty: the default one
HF_SPLITS = setting("HF_SPLITS", "train")              # the parts to read, separated by commas, for example "train,validation"
HF_REVISION = setting("HF_REVISION", "")               # a version of the dataset; empty: the newest
TEXT_FIELD = setting("TEXT_FIELD", "text")             # the column with the sentence
AUDIO_FIELD = setting("AUDIO_FIELD", "audio")          # the column with the recording
SPEAKER_FIELD = setting("SPEAKER_FIELD", "")           # the column with the speaker; empty: none
HF_PACE_EVERY = setting("HF_PACE_EVERY", 50)           # pause after this many rows while the notebook counts the hours
HF_PACE_SECONDS = setting("HF_PACE_SECONDS", 1.0)      # the length of that pause, in seconds

# ── CORPUS_KIND = "ljspeech": a folder with metadata.csv and wavs/ ──
LJSPEECH_DIR = setting("LJSPEECH_DIR", "")             # the folder
LJSPEECH_SPEAKER = setting("LJSPEECH_SPEAKER", "")     # the name of its one speaker; empty: none

# ── CORPUS_KIND = "csv": a table with a sentence and a recording on each row ──
CSV_PATH = setting("CSV_PATH", "")                     # the CSV file
CSV_TEXT_COL = setting("CSV_TEXT_COL", "text")         # the column with the sentence
CSV_AUDIO_COL = setting("CSV_AUDIO_COL", "audio")      # the column with the path of the recording
CSV_AUDIO_BASE_DIR = setting("CSV_AUDIO_BASE_DIR", "") # the folder those paths start from; empty: the folder of the CSV file
CSV_SPEAKER_COL = setting("CSV_SPEAKER_COL", "")       # the column with the speaker; empty: none
CSV_ID_COL = setting("CSV_ID_COL", "")                 # the column with a name for each row; empty: the row number

# ── Which speakers and how much ──
# Set at most one of SPEAKERS and TOP_SPEAKERS. Leave both empty to keep every speaker.
SPEAKERS = setting("SPEAKERS", "")                     # the speakers to keep, separated by commas
TOP_SPEAKERS = setting("TOP_SPEAKERS", 0)              # keep this many speakers with the most hours; 0: off
MAX_HOURS = setting("MAX_HOURS", 0.0)                  # stop after this many hours of recordings; 0: no limit

# ── Quick test ──
SMOKE = setting("SMOKE", False)                        # True: a quick test with a small built-in table of silent clips
N_SAMPLES = setting("N_SAMPLES", 4)                    # the number of good rows in that table


> ⚠ **`CORPUS_KIND=hf` needs outbound internet access** and a Hugging Face dataset download that can reach a few hundred megabytes on each of its two passes (accounting, then writing); that path is not run by CI.
> `SMOKE = True` (in the Settings cell) never reaches the network: it builds a tiny built-in two-speaker CSV fixture and ingests that, so the smoke path is CPU-only and fast.


---
## 0 · Check the settings

The settings are in the Settings cell at the top of the notebook. This cell checks them, and for a quick test it
makes a small table of silent clips to read.

The pipeline and the manifest format are described in
[docs/pipeline.md](../docs/pipeline.md) and [docs/manifest.md](../docs/manifest.md).


In [ ]:
import os


def _build_smoke_fixture(n_samples):
    """A tiny built-in CSV fixture: n_samples rows mixing text that passes
    validation with text that Stage 4 refuses (bracketed, no letter), split
    across two speakers so the speaker-selection stage (Stage 3) has something
    real to select between. Ingest never reads what a clip says, only its text
    and its channel count, so silence exercises the same code paths as real
    speech.
    """
    import csv
    import tempfile
    import numpy as np
    import soundfile as sf

    # (text, speaker): speaker selection (Stage 3) runs before text validation
    # (Stage 4), so a row from an excluded speaker never reaches the text
    # check. spk_b's row carries valid text on purpose -- if speaker selection
    # ever stopped filtering, that row would leak into the output and the
    # assertion in Stage 5 would catch it. These three always run, regardless
    # of n_samples, so the SMOKE assertions in Stage 5 always have something to
    # find; only the passing spk_a row count scales with n_samples.
    fixed_rows = [
        ("Another perfectly fine line of text.", "spk_b"),  # excluded by speaker selection
        ("[inaudible]", "spk_a"),   # refused: bracketed annotation
        ("12345", "spk_a"),        # refused: no letter
    ]
    n_valid = max(int(n_samples), 1)
    rows = [("Hello there, this is a valid sentence.", "spk_a")] * n_valid + fixed_rows

    fixture_dir = Path(tempfile.mkdtemp(prefix="tts_corpus_ingest_smoke_"))
    (fixture_dir / "wavs").mkdir()
    csv_path = fixture_dir / "corpus.csv"
    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "text", "speaker", "audio"])
        for i, (text, speaker) in enumerate(rows):
            row_id = f"smoke{i + 1:03d}"
            audio_name = f"wavs/{row_id}.wav"
            sf.write(str(fixture_dir / audio_name), np.zeros(8000, dtype="float32"), 16000, subtype="PCM_16")
            writer.writerow([row_id, text, speaker, audio_name])
    return str(csv_path)


if SMOKE:
    from pathlib import Path
    CORPUS_KIND = "csv"
    CSV_PATH = _build_smoke_fixture(N_SAMPLES)
    CSV_TEXT_COL = "text"
    CSV_AUDIO_COL = "audio"
    CSV_SPEAKER_COL = "speaker"
    CSV_ID_COL = "id"
    CORPUS_NAME = CORPUS_NAME or "smoke"
    SPEAKERS = SPEAKERS or "spk_a"   # exercise speaker selection: only spk_a is kept
    print(f"SMOKE=True: ingesting the built-in fixture at {CSV_PATH}")


def _default_output_dir():
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return "/kaggle/working/corpus_ingest_out"
    if os.path.isdir("/content"):
        return "/content/corpus_ingest_out"
    return "./corpus_ingest_out"


OUTPUT_DIR = OUTPUT_DIR or _default_output_dir()

# The Hugging Face download cache: to put it on a disk with room, set the environment variable HF_HUB_CACHE
# before you start the notebook. Never set HF_HOME for this, because it also moves the Hugging Face login.

assert CORPUS_KIND in ("hf", "ljspeech", "csv"), f"CORPUS_KIND must be hf, ljspeech or csv, got {CORPUS_KIND!r}"

print("Configuration loaded.")
print(f"  Corpus kind    : {CORPUS_KIND}")
print(f"  Language       : {CORPUS_LANG}")
print(f"  Output dir     : {OUTPUT_DIR}")
if CORPUS_KIND == "hf":
    print(f"  HF dataset     : {HF_DATASET}  config={HF_CONFIG or '(default)'}  splits={HF_SPLITS}")
    print(f"  Fields         : text={TEXT_FIELD}  audio={AUDIO_FIELD}  speaker={SPEAKER_FIELD or '(none)'}")
elif CORPUS_KIND == "ljspeech":
    print(f"  LJSpeech dir   : {LJSPEECH_DIR}  speaker={LJSPEECH_SPEAKER or '(none)'}")
else:
    print(f"  CSV path       : {CSV_PATH}  text={CSV_TEXT_COL}  audio={CSV_AUDIO_COL}  speaker={CSV_SPEAKER_COL or '(none)'}")
print(f"  Speaker rule   : SPEAKERS={SPEAKERS or '(none)'}  TOP_SPEAKERS={TOP_SPEAKERS}  MAX_HOURS={MAX_HOURS or '(no limit)'}")


---
## 1 · Load corpus rows

Each row is normalised to the same shape regardless of `CORPUS_KIND`: an id, a text, an optional
speaker, and either a decoded audio array (`CORPUS_KIND=hf`) or a path to a local WAV
(`CORPUS_KIND=ljspeech` or `csv`). The `hf` reader pulls one Hub shard at a time in streaming mode
and paces itself; the other two readers read local files directly.

A Hugging Face row is keyed `<dataset>:<split>:<row index>` — the row's position inside its own
split, never a bare position across splits. An `ljspeech` or `csv` row is keyed
`<corpus name>:<row id>`.

| Setting | Meaning |
|---|---|
| `CORPUS_KIND` | `hf`, `ljspeech` or `csv` |
| `HF_DATASET`, `HF_CONFIG`, `HF_SPLITS`, `HF_REVISION` | Which Hugging Face dataset, config, splits and revision |
| `TEXT_FIELD`, `AUDIO_FIELD`, `SPEAKER_FIELD` | Column names for `CORPUS_KIND=hf` |
| `LJSPEECH_DIR`, `LJSPEECH_SPEAKER` | Directory and speaker id for `CORPUS_KIND=ljspeech` |
| `CSV_PATH`, `CSV_TEXT_COL`, `CSV_AUDIO_COL`, `CSV_AUDIO_BASE_DIR`, `CSV_SPEAKER_COL`, `CSV_ID_COL` | Columns for `CORPUS_KIND=csv` |
| `HF_HUB_CACHE`, `HF_PACE_EVERY`, `HF_PACE_SECONDS` | Hub cache location and download pacing |


In [ ]:
%pip install --quiet --pre "datasets>=5.0.1" "huggingface_hub>=0.36.0,<2.0" "soundfile>=0.14.0" "numpy>=2.0"
# huggingface_hub is capped below 2.0 here because datasets>=5.0.1 itself requires huggingface_hub<2.0;
# a higher floor on this line alone would make the install unresolvable.


In [ ]:
import csv
import json
import re
import time
from pathlib import Path

import soundfile as sf


def _corpus_source_meta():
    if CORPUS_KIND == "hf":
        name = CORPUS_NAME or HF_DATASET
        url = CORPUS_URL or f"https://huggingface.co/datasets/{HF_DATASET}"
    elif CORPUS_KIND == "ljspeech":
        name = CORPUS_NAME or Path(LJSPEECH_DIR).name
        url = CORPUS_URL
    else:
        name = CORPUS_NAME or Path(CSV_PATH).stem
        url = CORPUS_URL
    return name, url


CORPUS_NAME, CORPUS_URL = _corpus_source_meta()
HF_RESOLVED_REVISION = None  # filled in by _resolve_hf_revision() for CORPUS_KIND=hf


def _resolve_hf_revision():
    """The Hub commit sha actually read. Recorded in the .meta.json sidecar."""
    global HF_RESOLVED_REVISION
    from huggingface_hub import HfApi
    api = HfApi()
    info = api.dataset_info(HF_DATASET, revision=HF_REVISION or None)
    HF_RESOLVED_REVISION = info.sha
    return HF_RESOLVED_REVISION


def _iter_hf_rows(pace=False):
    import io
    from datasets import load_dataset
    from datasets.features import Audio

    dataset = HF_DATASET
    config = HF_CONFIG or None
    revision = HF_RESOLVED_REVISION or (HF_REVISION or None)
    text_field = TEXT_FIELD
    audio_field = AUDIO_FIELD
    speaker_field = SPEAKER_FIELD
    pace_every = HF_PACE_EVERY
    pace_seconds = HF_PACE_SECONDS or 0.0
    splits = [s.strip() for s in HF_SPLITS.split(",") if s.strip()]

    for split in splits:
        ds = load_dataset(dataset, config, split=split, streaming=True, revision=revision)
        # decode=False: read each file's own bytes and decode them with soundfile below,
        # instead of the library's own audio decoder (which needs torch/torchcodec and
        # can resample). This is also how the notebook keeps every row at its native rate.
        ds = ds.cast_column(audio_field, Audio(decode=False))
        for i, row in enumerate(ds):
            if pace and pace_every and i > 0 and i % pace_every == 0 and pace_seconds:
                time.sleep(pace_seconds)
            key = f"{dataset}:{split}:{i}"
            audio_bytes = row[audio_field]["bytes"]
            assert audio_bytes is not None, (
                f"{key}: this dataset stores audio as an external path, not embedded bytes; "
                "add a reader branch for that case before using it here"
            )
            arr, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32", always_2d=False)
            yield {
                "id": key,
                "text": row.get(text_field, "") or "",
                "speaker": row.get(speaker_field) if speaker_field else None,
                "audio_array": arr,
                "sample_rate": int(sr),
                "source": {"kind": "corpus", "name": CORPUS_NAME, "url": CORPUS_URL, "item": key},
            }


def _read_ljspeech_metadata_line(line):
    """id|text or id|text|normalized. The last field is the text of record."""
    parts = line.rstrip("\n").split("|")
    row_id = parts[0]
    text = parts[2] if len(parts) > 2 else parts[1]
    return row_id, text


def _iter_ljspeech_rows(pace=False):
    ljdir = Path(LJSPEECH_DIR)
    speaker = LJSPEECH_SPEAKER or None
    with open(ljdir / "metadata.csv", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row_id, text = _read_ljspeech_metadata_line(line)
            key = f"{CORPUS_NAME}:{row_id}"
            yield {
                "id": key,
                "text": text,
                "speaker": speaker,
                "audio_path": str(ljdir / "wavs" / f"{row_id}.wav"),
                "source": {"kind": "corpus", "name": CORPUS_NAME, "url": CORPUS_URL, "item": row_id},
            }


def _iter_csv_rows(pace=False):
    csv_path = Path(CSV_PATH)
    base_dir = Path(CSV_AUDIO_BASE_DIR or csv_path.parent)
    text_col = CSV_TEXT_COL
    audio_col = CSV_AUDIO_COL
    speaker_col = CSV_SPEAKER_COL or None
    id_col = CSV_ID_COL or None
    with open(csv_path, encoding="utf-8") as f:
        for i, row in enumerate(csv.DictReader(f)):
            row_id = row[id_col] if id_col else str(i)
            key = f"{CORPUS_NAME}:{row_id}"
            yield {
                "id": key,
                "text": row.get(text_col, "") or "",
                "speaker": row.get(speaker_col) if speaker_col else None,
                "audio_path": str(base_dir / row[audio_col]),
                "source": {"kind": "corpus", "name": CORPUS_NAME, "url": CORPUS_URL, "item": row_id},
            }


def iter_corpus_rows(pace=False):
    if CORPUS_KIND == "hf":
        yield from _iter_hf_rows(pace=pace)
    elif CORPUS_KIND == "ljspeech":
        yield from _iter_ljspeech_rows(pace=pace)
    else:
        yield from _iter_csv_rows(pace=pace)


def _row_duration_s(row):
    """Cheap duration: a soundfile header read for a path, the array length for a decoded row."""
    if "audio_array" in row:
        return len(row["audio_array"]) / row["sample_rate"]
    info = sf.info(row["audio_path"])
    return info.frames / info.samplerate


def _load_audio_native(row):
    """The full audio at the corpus's own sample rate. This notebook never resamples."""
    if "audio_array" in row:
        return row["audio_array"], row["sample_rate"]
    arr, sr = sf.read(row["audio_path"], dtype="float32", always_2d=False)
    return arr, sr


if CORPUS_KIND == "hf":
    _resolve_hf_revision()
    print(f"Reading {HF_DATASET} at revision {HF_RESOLVED_REVISION}")


---
## 2 · Speaker accounting (first pass)

A first pass reads every row once to sum audio duration and count rows per speaker, without
keeping the audio in memory. Every distinct speaker value is reported with its row count; a value
that is empty or reads like a bucket (`unknown`, `other`, `multiple`) is named here and left out of
the `TOP_SPEAKERS` ranking below, since it does not identify one speaker.


In [ ]:
from collections import Counter

_BUCKET_SPEAKER_VALUES = {"", "unknown", "other", "multiple"}


def _speaker_key(row):
    sp = row["speaker"]
    return "" if sp is None else str(sp).strip()


def _is_bucket_speaker(sp):
    return sp.lower() in _BUCKET_SPEAKER_VALUES


speaker_rows = Counter()
speaker_seconds = Counter()
n_scanned = 0

for row in iter_corpus_rows(pace=True):
    n_scanned += 1
    sp = _speaker_key(row)
    speaker_rows[sp] += 1
    speaker_seconds[sp] += _row_duration_s(row)

print(f"Rows scanned: {n_scanned}")
print(f"{'speaker':30s} {'rows':>8} {'hours':>10}")
for sp, n in sorted(speaker_rows.items(), key=lambda kv: -kv[1]):
    label = sp if sp else "(empty)"
    print(f"{label:30s} {n:>8} {speaker_seconds[sp] / 3600:>10.4f}")

bucket_values = sorted(sp for sp in speaker_rows if _is_bucket_speaker(sp))
if bucket_values:
    print(f"\nExcluded from TOP_SPEAKERS (bucket values): {[v or '(empty)' for v in bucket_values]}")

ranked_speakers = sorted(
    ((sp, speaker_rows[sp], speaker_seconds[sp]) for sp in speaker_rows if not _is_bucket_speaker(sp)),
    key=lambda t: -t[2],
)
print(f"\nTop {min(20, len(ranked_speakers))} speakers by hours:")
print(f"{'speaker':30s} {'hours':>10} {'rows':>8}")
for sp, n, secs in ranked_speakers[:20]:
    print(f"{sp:30s} {secs / 3600:>10.4f} {n:>8}")


---
## 3 · Speaker selection

`SPEAKERS` names an explicit list of speakers to keep. `TOP_SPEAKERS` keeps the N speakers with
the most audio from the ranking above, bucket values excluded. Set at most one; leaving both empty
(or `TOP_SPEAKERS=0`) keeps every speaker — the right setting for a corpus that names none
(`ljspeech` with no `LJSPEECH_SPEAKER`, or `hf`/`csv` with no `SPEAKER_FIELD`/`CSV_SPEAKER_COL`).

| Setting | Meaning |
|---|---|
| `SPEAKERS` | Comma list of speaker ids to keep |
| `TOP_SPEAKERS` | Keep the N speakers with the most audio, 0 = off |


In [ ]:
_speakers_cfg = [s.strip() for s in SPEAKERS.split(",") if s.strip()]
_top_n = TOP_SPEAKERS

assert not (_speakers_cfg and _top_n), "set at most one of SPEAKERS or TOP_SPEAKERS"

if _speakers_cfg:
    keep_speakers = set(_speakers_cfg)
    _missing_speakers = sorted(set(_speakers_cfg) - set(speaker_rows))
    if _missing_speakers:
        raise ValueError(
            f"SPEAKERS names {_missing_speakers} not seen in the corpus (case-sensitive); "
            f"speakers seen: {sorted(sp or '(empty)' for sp in speaker_rows)}"
        )
    speaker_rule = {"mode": "explicit_list", "speakers": sorted(keep_speakers)}
elif _top_n:
    keep_speakers = {sp for sp, _, _ in ranked_speakers[:_top_n]}
    speaker_rule = {"mode": "top_n", "n": _top_n, "speakers": sorted(keep_speakers)}
else:
    keep_speakers = None
    speaker_rule = {"mode": "all"}

print(f"Speaker rule: {speaker_rule}")
if keep_speakers is not None:
    kept_rows = sum(speaker_rows[sp] for sp in keep_speakers)
    print(f"Rows from kept speakers, before text validation: {kept_rows} of {n_scanned}")


---
## 4 · Text validation

A row is refused when its text is empty, contains a bracketed annotation (`[...]` or `<...>`), or
has no letter. Ingest only refuses a bad row; it never rewrites the text, and it computes no
quality score here — scoring is the measure notebook's job, not ingest's.


---
## 5 · Write the manifest

A second pass walks the corpus again, in the order it yields rows, and for each row that passes
the speaker rule and text validation: decodes its audio, asserts it is mono (a multi-channel file
is downmixed by averaging its channels), and writes it as PCM WAV at its own native sample rate.
This notebook never resamples; if it ever needed to, the row's `.meta.json` entry would name the
rate it converted from. Once the accumulated duration reaches `MAX_HOURS`, further rows are
counted as capped rather than written.

| Setting | Meaning |
|---|---|
| `MAX_HOURS` | Cap on the hours written, 0 = no cap |
| `OUTPUT_DIR`, `MANIFEST_NAME` | Where the manifest, its sidecar and `wavs/` are written |


In [ ]:
import shutil

_BRACKET_RE = re.compile(r"\[[^\]]*\]|<[^>]*>")


def _text_refusal_reason(text):
    if not text or not text.strip():
        return "empty"
    if _BRACKET_RE.search(text):
        return "bracketed_annotation"
    if not any(ch.isalpha() for ch in text):
        return "no_letter"
    return None


def _slug(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)


max_hours_cfg = MAX_HOURS or None
max_seconds = max_hours_cfg * 3600 if max_hours_cfg else None

out_dir = Path(OUTPUT_DIR)
wavs_dir = out_dir / "wavs"
_marker = out_dir / ".created_by_tts_corpus_ingest"
if wavs_dir.exists() and any(wavs_dir.iterdir()):
    # A write pass never mixes with older WAVs, and never deletes a folder:
    # an earlier run of this notebook is moved aside, anything else stops the run.
    if not _marker.exists():
        raise RuntimeError(
            f"{wavs_dir} already holds files that this notebook did not write. "
            "Set OUTPUT_DIR to an empty or new directory."
        )
    _aside = out_dir / f"wavs.old-{int(time.time())}"
    wavs_dir.rename(_aside)
    print(f"Moved the previous run's wavs/ aside to {_aside.name}")
wavs_dir.mkdir(parents=True, exist_ok=True)
_marker.touch()

reason_counts = Counter()
manifest_rows = []
downmixed_ids = []
n_seen = 0
n_speaker_filtered_out = 0
n_capped = 0
total_seconds = 0.0

for row in iter_corpus_rows(pace=False):
    n_seen += 1
    sp = _speaker_key(row)
    if keep_speakers is not None and sp not in keep_speakers:
        n_speaker_filtered_out += 1
        continue

    reason = _text_refusal_reason(row["text"])
    if reason:
        reason_counts[reason] += 1
        continue

    arr, sr = _load_audio_native(row)
    n_frames_native = arr.shape[0]
    if arr.ndim > 1:
        arr = arr.mean(axis=-1)
        downmixed_ids.append(row["id"])
    assert arr.ndim == 1 and arr.shape[0] == n_frames_native, (
        f"{row['id']}: downmix changed the frame count or left more than one dimension"
    )

    dur = len(arr) / sr
    if max_seconds is not None and total_seconds + dur > max_seconds:
        n_capped += 1
        continue
    total_seconds += dur

    out_name = f"{_slug(row['id'])}.wav"
    sf.write(str(wavs_dir / out_name), arr, sr, subtype="PCM_16")

    row_out = {
        "id": row["id"],
        "text": row["text"],
        "lang": CORPUS_LANG,
        "audio": f"wavs/{out_name}",
        "duration_s": round(dur, 3),
        "sample_rate": int(sr),
        "source": row["source"],
    }
    if sp:
        row_out["speaker"] = sp
    manifest_rows.append(row_out)
    reason_counts["passed"] += 1

print(f"Rows seen in the write pass  : {n_seen}")
print(f"Filtered out by speaker rule : {n_speaker_filtered_out}")
print("Text validation:")
for reason in ("empty", "bracketed_annotation", "no_letter", "passed"):
    print(f"  {reason:22s} {reason_counts.get(reason, 0)}")
print(f"Capped by MAX_HOURS          : {n_capped}")
print(f"Downmixed to mono            : {len(downmixed_ids)}")
print(f"Rows written                 : {len(manifest_rows)}  ({total_seconds / 3600:.4f} hours)")

if not manifest_rows:
    raise RuntimeError(
        f"No rows written: 0 of {n_seen} rows in the write pass survived speaker selection "
        f"and text validation. SPEAKERS={SPEAKERS or '(unset)'}, speaker_rule="
        f"{speaker_rule}; speakers seen in the corpus: "
        f"{sorted(sp or '(empty)' for sp in speaker_rows)}."
    )

if SMOKE:
    # The built-in fixture (Stage 0) exists to show two things working: text
    # refusal and speaker selection. Neither is worth anything unless a broken
    # version of either would show up here.
    assert reason_counts.get("bracketed_annotation", 0) >= 1, (
        "the smoke fixture's bracketed-annotation row was not refused"
    )
    assert reason_counts.get("no_letter", 0) >= 1, (
        "the smoke fixture's letterless row was not refused"
    )
    assert n_speaker_filtered_out >= 1, (
        "the smoke fixture's excluded-speaker row (valid text, wrong speaker) "
        "was not filtered by speaker selection"
    )
    _kept_speakers = {r.get("speaker") for r in manifest_rows}
    assert _kept_speakers <= {"spk_a"}, (
        f"a row from an excluded speaker survived selection: {_kept_speakers}"
    )


---
## 6 · Manifest, sidecar and summary


In [ ]:
manifest_path = out_dir / MANIFEST_NAME
with open(manifest_path, "w", encoding="utf-8") as f:
    for row in manifest_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

meta = {
    "notebooks": ["tts_corpus_ingest"],
    "corpus": {
        "kind": CORPUS_KIND,
        "id": (
            HF_DATASET if CORPUS_KIND == "hf"
            else os.path.basename(LJSPEECH_DIR.rstrip("/\\")) if CORPUS_KIND == "ljspeech"
            else os.path.basename(CSV_PATH)
        ),
        "name": CORPUS_NAME,
        "config": (HF_CONFIG or None) if CORPUS_KIND == "hf" else None,
        "splits": ([s.strip() for s in HF_SPLITS.split(",") if s.strip()]
                   if CORPUS_KIND == "hf" else None),
        "revision": HF_RESOLVED_REVISION if CORPUS_KIND == "hf" else None,
    },
    "speaker_rule": speaker_rule,
    "excluded_bucket_speaker_values": [v or "(empty)" for v in bucket_values],
    "counts": {
        "rows_scanned": n_scanned,
        "rows_speaker_filtered_out": n_speaker_filtered_out,
        "rows_text_refused": {k: v for k, v in reason_counts.items() if k != "passed"},
        "rows_capped_by_max_hours": n_capped,
        "rows_written": len(manifest_rows),
        "hours_written": round(total_seconds / 3600, 4),
        "max_hours_cap": max_hours_cfg,
        "downmixed_to_mono": len(downmixed_ids),
    },
}

meta_path = out_dir / (MANIFEST_NAME + ".meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"Manifest : {manifest_path}  ({len(manifest_rows)} rows)")
print(f"Sidecar  : {meta_path}")
print(f"Audio    : {wavs_dir}")
if manifest_rows:
    print("First row:", json.dumps(manifest_rows[0], ensure_ascii=False))


[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the [NGI0 Commons Fund](https://nlnet.nl/commonsfund), a fund established by [NLnet](https://nlnet.nl) with financial support from the European Commission's [Next Generation Internet](https://ngi.eu) programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No [101135429](https://cordis.europa.eu/project/id/101135429).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Licensed under Apache-2.0.